Blog Generator using prompt chaining (Sequential LLM workflow)

In [58]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict
import os


In [59]:
load_dotenv()

llm=ChatOpenAI()

In [60]:
#define State
class BlogState(TypedDict):
    topic:str
    outline:str
    blog:str
    blog_evaluation_feedback:str


In [61]:
def generate_outline(state : BlogState)-> BlogState :
    """This function generates an short precise outline for the topic provided""" 
    topic = state['topic']
    prompt = f"""
        Generate a short precise outline for the given topic {topic} 
    """
    result = llm.invoke(prompt)
    state['outline'] = result.content
    return state

def generate_blog(state:BlogState)->BlogState :
    """This function generates a concise inpactful blog based on given topic and its outline""" 

    outline = state['outline']
    topic = state['topic']
    prompt = f"""
        Generate a concise impactful blog using outline {outline} for the given topic {topic} 
    """
    result = llm.invoke(prompt)
    state['blog'] = result.content
    return state

def evaluate_blog(state:BlogState)->BlogState :
    """This function evaluates the blog against teh outline and generate a remarks on quality""" 
    outline = state['outline']
    blog = state['blog']
    prompt = f"""
        Evaluate the blog {blog} against the outline {outline}, check the standard and whether it follows the outline, anf finally generate a evaluation feedback 
    """
    result = llm.invoke(prompt)
    state['blog_evaluation_feedback'] = result.content
    return state



In [62]:
#define graph
graph = StateGraph(BlogState)

#define nodes
graph.add_node('generate_outline', generate_outline)
graph.add_node('generate_blog', generate_blog)
graph.add_node('evaluate_blog', evaluate_blog)

#define edges
graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_blog')
graph.add_edge('generate_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)



In [63]:
workflow = graph.compile()


In [ ]:
input_state = {'topic':'Modern Agentic AI'}
output_state =workflow.invoke(input_state)
print(output_state['topic'])

print(output_state['outline'])

print(output_state['blog'])

print(output_state['blog_evaluation_feedback'])



Modern Agentic AI
I. Introduction
   A. Definition of modern agentic AI
   B. Importance and relevance in today's society

II. Characteristics of modern agentic AI
   A. Ability to make autonomous decisions and take action
   B. Capacity to learn and adapt to new information
   C. Interaction with its environment and users
   D. Potential risks and ethical considerations

III. Applications of modern agentic AI
   A. Autonomous vehicles
   B. Smart home devices
   C. Healthcare technologies
   D. Customer service chatbots

IV. Advantages and benefits of modern agentic AI
   A. Increased efficiency and productivity
   B. Ability to handle complex tasks and data analysis
   C. Improved decision-making processes
   D. Enhanced user experience and personalization

V. Challenges and limitations of modern agentic AI
   A. Concerns about privacy and security
   B. Potential for bias and discrimination
   C. Lack of transparency and explainability
   D. Need for ongoing regulation and oversight